In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import math
import time

CATALOG = "fahrlytics"
SILVER  = f"{CATALOG}.silver"
GOLD    = f"{CATALOG}.gold"

spark.sql(f"USE CATALOG {CATALOG}")

print(f"Catalog           : {CATALOG}")
print(f"Source Table      : {SILVER}.trips")
print(f"Target Gold Table : {GOLD}.trip_features")

In [0]:
print("Reading clean trips from Silver...")

df_silver_trips = (
    spark.table(f"{SILVER}.trips")
    .filter(F.col("is_clean_record") == True)
    .withColumn("pickup_hour", F.hour("pickup_time"))
)

In [0]:
print("Aggregating trips to city-date-hour grain...")

hourly_aggregates = (
    df_silver_trips.groupBy("city", "trip_date", "pickup_hour", "day_type")
    .agg(
        # Target Demand
        F.count(F.when(F.col("trip_status") == "completed", 1)).alias("trip_count"),
        F.count(F.when(F.col("trip_status") == "cancelled", 1)).alias("cancelled_trip_count"),
        F.count("trip_id").alias("total_demand_requests"),

        # Operational & Price Signals
        F.round(F.avg("surge_multiplier"), 2).alias("avg_surge_multiplier"),
        F.round(F.max("surge_multiplier"), 2).alias("max_surge_multiplier"),
        F.round(F.avg(F.when(F.col("trip_status") == "completed", F.col("fare_amount"))), 2).alias("avg_fare_eur"),
        F.round(F.sum(F.when(F.col("trip_status") == "completed", F.col("fare_amount"))), 2).alias("total_revenue_eur"),
        F.round(F.avg("distance_km"), 2).alias("avg_distance_km"),
        F.round(F.avg("trip_duration_min"), 2).alias("avg_duration_min"),

        # Fleet Mix Proportions
        F.count(F.when(F.col("vehicle_type") == "Standard", 1)).alias("fleet_standard_trips"),
        F.count(F.when(F.col("vehicle_type") == "Comfort", 1)).alias("fleet_comfort_trips"),
        F.count(F.when(F.col("vehicle_type") == "Van/XL", 1)).alias("fleet_van_trips"),

        # Weather Mode (predominant hourly weather condition)
        F.expr("mode(weather)").alias("weather_condition")
    )
    .withColumn(
        "cancellation_rate",
        F.round(F.col("cancelled_trip_count") / F.greatest(F.col("total_demand_requests"), F.lit(1)), 3)
    )
)

In [0]:
print("Adding calendar features and cyclical trigonometry encodings...")

two_pi = 2.0 * math.pi

df_calendar = (
    hourly_aggregates
    .withColumn("day_of_week", F.dayofweek("trip_date")) # 1 = Sun, 7 = Sat
    .withColumn("month", F.month("trip_date"))
    .withColumn("is_weekend", F.col("day_type").isin("weekend", "holiday"))
    # Cyclical hour encoding preserves the continuity between 23:00 and 00:00
    .withColumn("hour_sin", F.round(F.sin(F.col("pickup_hour") * (two_pi / 24.0)), 4))
    .withColumn("hour_cos", F.round(F.cos(F.col("pickup_hour") * (two_pi / 24.0)), 4))
    # Cyclical day of week encoding (1-7 scale)
    .withColumn("dow_sin", F.round(F.sin(F.col("day_of_week") * (two_pi / 7.0)), 4))
    .withColumn("dow_cos", F.round(F.cos(F.col("day_of_week") * (two_pi / 7.0)), 4))
)

In [0]:
print("Generating sequential lag signals (1h, 24h, 168h)...")

# Construct ordered hourly window within each city
city_time_window = Window.partitionBy("city").orderBy("trip_date", "pickup_hour")

df_features = (
    df_calendar
    # Lag 1: Preceding hour demand
    .withColumn("demand_lag_1h", F.lag("trip_count", 1).over(city_time_window))
    # Lag 24: Same hour yesterday
    .withColumn("demand_lag_24h", F.lag("trip_count", 24).over(city_time_window))
    # Lag 168: Same hour 7 days ago (168 hours in a week)
    .withColumn("demand_lag_168h", F.lag("trip_count", 168).over(city_time_window))
    # 24-Hour Rolling Demand Average (historical baseline)
    .withColumn("rolling_avg_demand_24h", 
        F.round(F.avg("trip_count").over(city_time_window.rowsBetween(-24, -1)), 2)
    )
    # Fill startup boundary nulls with current hour value
    .withColumn("demand_lag_1h", F.coalesce(F.col("demand_lag_1h"), F.col("trip_count")))
    .withColumn("demand_lag_24h", F.coalesce(F.col("demand_lag_24h"), F.col("trip_count")))
    .withColumn("demand_lag_168h", F.coalesce(F.col("demand_lag_168h"), F.col("trip_count")))
    .withColumn("rolling_avg_demand_24h", F.coalesce(F.col("rolling_avg_demand_24h"), F.col("trip_count")))
    .withColumn("_feature_calculated_at", F.current_timestamp())
)

In [0]:
print(f"Writing to {GOLD}.trip_features...")
t0 = time.time()

(
    df_features.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("city")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD}.trip_features")
)

print(f"Gold trip features written in {round(time.time() - t0, 2)}s.")

print("Running OPTIMIZE with Z-ORDER on trip_date and pickup_hour...")
spark.sql(f"OPTIMIZE {GOLD}.trip_features ZORDER BY (trip_date, pickup_hour)")
print("OPTIMIZE completed.")

In [0]:
print("=== FAHRLYTICS GOLD TRIP FEATURES AUDIT ===\n")

df_gold_trips = spark.table(f"{GOLD}.trip_features")
total_rows = df_gold_trips.count()
print(f"Total city-hourly intervals: {total_rows:,}\n")

print("City Volume & Demand Summary:")
df_gold_trips.groupBy("city").agg(
    F.count("trip_count").alias("total_hourly_bins"),
    F.sum("trip_count").alias("total_completed_trips"),
    F.round(F.avg("trip_count"), 1).alias("avg_hourly_demand"),
    F.round(F.max("trip_count"), 0).alias("peak_hourly_demand")
).orderBy(F.col("total_completed_trips").desc()).show()

print("Time-Series Sample (Berlin peak hours):")
df_gold_trips.filter((F.col("city") == "Berlin") & (F.col("pickup_hour").isin(8, 9, 18, 19))) \
    .select(
        "city", "trip_date", "pickup_hour", "weather_condition",
        "trip_count", "demand_lag_1h", "demand_lag_24h", "avg_surge_multiplier"
    ) \
    .orderBy("trip_date", "pickup_hour") \
    .show(8, truncate=False)